# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prakritibhandari07/FlyRank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Prakritibhandari07/FlyRank-ml-internship"
REPO_DIR = "FlyRank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check repo/path"
print("Starter data found. You're ready.")

Working dir: /content/FlyRank-ml-internship/FlyRank-ml-internship/FlyRank-ml-internship
Starter data found. You're ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page (content_id) in the starter snapshot. The starter CSV contains one current snapshot per content page; it does not include a date/month column, so the exact calendar date range cannot be verified from this file. I will use the full warehouse release for the required month-specific checks.

In [17]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Total rows: {df.shape[0]}")
print(f"Unique content_id values: {df['content_id'].nunique()}")
print(
    f"Rows match unique IDs (no duplicates): "
    f"{df.shape[0] == df['content_id'].nunique()}"
)

Total rows: 30000
Unique content_id values: 30000
Rows match unique IDs (no duplicates): True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label:** `trend_direction` (specifically, `is_declining_label = (trend_direction == "down")`) — this is the proxy target.

**Features:** `impressions_90d`, `days_since_last_update`, `avg_position`, `ctr`, `sessions_90d` — all observable signals available before the decision moment.

**Context (not used as features):** `content_id`, `client_id` — identifiers used for grouping and deduplication, not predictive signals.

**Excluded:** `trend_direction` and `trend_pct` — excluded because they are label-derived. Using them as features would leak information about the target and could produce an unrealistically strong result.

In [18]:
print("Columns available in dataset:")
print(list(df.columns))

Columns available in dataset:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [19]:

# Query 1 — Grain

total_rows = len(df)
unique_content_ids = df["content_id"].nunique()

print("Total rows:", total_rows)
print("Unique content_id values:", unique_content_ids)
print("One row per content_id:", total_rows == unique_content_ids)

assert total_rows == unique_content_ids
# Query 2 — Row count and date/month availability


print("Row count:", len(df))
print("Time window: Current starter snapshot")
print("Date/month field: Not available in the starter CSV")

# Query 3 — Availability
availability_columns = [
    c for c in df.columns
    if any(word in c.lower() for word in ["available", "availability"])
]

print("Availability columns in starter CSV:", availability_columns)
print("The required IS TRUE availability check must be run on the warehouse release.")

## Five-feature frame
features = [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "sessions_90d"
]

feature_frame = df[features].copy()

feature_frame["is_declining_label"] = (
    df["trend_direction"] == "down"
)

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())



Total rows: 30000
Unique content_id values: 30000
One row per content_id: True
Row count: 30000
Time window: Current starter snapshot
Date/month field: Not available in the starter CSV
Availability columns in starter CSV: []
The required IS TRUE availability check must be run on the warehouse release.
Feature frame shape: (30000, 6)


,impressions_90d,days_since_last_update,avg_position,ctr,sessions_90d,is_declining_label
0,3803,20,10.6,0.76,17,True
1,15320,25,20.3,0.05,9,True
2,12581,20,36.5,0.09,11,True
3,11751,22,6.2,0.49,78,False
4,19140,14,44.0,0.13,145,True


Verifying the claims above with real numbers.

### Feature availability

- **`impressions_90d`** — Knowable at the decision moment because it summarizes impressions already observed during the preceding 90-day window.

- **`days_since_last_update`** — Knowable at the decision moment because it records how long the content has gone without an update.

- **`avg_position`** — Knowable at the decision moment because it summarizes observed search-position data available before the decision.

- **`ctr`** — Knowable at the decision moment because it is calculated from observed search interactions.

- **`sessions_90d`** — Knowable at the decision moment because it summarizes sessions already observed during the preceding 90-day window.

## Deliberate label leakage experiment

I will deliberately add a label-derived feature to demonstrate how data leakage can make a model score unrealistically well. I will then remove the leaked feature and keep the honest result.

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = feature_frame[features]
y = feature_frame["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_imp, y_train)

honest_score = accuracy_score(
    y_test,
    model.predict(X_test_imp)
)

print("Honest score:", round(honest_score, 3))

Honest score: 0.561


### Deliberate leakage

To demonstrate label leakage, I will add a feature that directly contains the target label. This feature is intentionally invalid and will be removed after the experiment.

In [21]:
# Deliberate leakage experiment

leaky_df = feature_frame.copy()

# This is intentionally leaked information.
leaky_df["label_leak"] = leaky_df["is_declining_label"].astype(int)

X_leaky = leaky_df[features + ["label_leak"]]
y_leaky = leaky_df["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y_leaky,
    test_size=0.2,
    random_state=42,
    stratify=y_leaky
)

imputer_leaky = SimpleImputer(strategy="median")

X_train_imp = imputer_leaky.fit_transform(X_train)
X_test_imp = imputer_leaky.transform(X_test)

leaky_model = LogisticRegression(max_iter=1000, random_state=42)

leaky_model.fit(X_train_imp, y_train)

leaky_score = accuracy_score(
    y_test,
    leaky_model.predict(X_test_imp)
)

print("Honest score:", round(honest_score, 3))
print("Leaky score:", round(leaky_score, 3))

Honest score: 0.561
Leaky score: 1.0


### Remove the leaked feature

The leaked feature is removed because it is derived directly from the target. The final feature set contains only the five honest features defined in the contract.

In [22]:
# Remove the leaked feature and evaluate the honest feature set again

X_clean = leaky_df[features]
y_clean = leaky_df["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y_clean,
    test_size=0.2,
    random_state=42,
    stratify=y_clean
)

imputer_clean = SimpleImputer(strategy="median")

X_train_imp = imputer_clean.fit_transform(X_train)
X_test_imp = imputer_clean.transform(X_test)

clean_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

clean_model.fit(X_train_imp, y_train)

clean_score = accuracy_score(
    y_test,
    clean_model.predict(X_test_imp)
)

print("Original honest score:", round(honest_score, 3))
print("Leaky score:", round(leaky_score, 3))
print("Clean score after removing leakage:", round(clean_score, 3))

Original honest score: 0.561
Leaky score: 1.0
Clean score after removing leakage: 0.561


### Leakage lesson

The deliberately label-derived feature caused an artificially strong score because it directly revealed the target to the model. After removing the leaked feature, the score returned to the honest baseline. The leaked feature is not included in the final feature set.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This starter dataset has real limits I need to keep in mind:

1. **The label is a proxy, not a future outcome.** `trend_direction` is
   calculated from the current window, not a decision point followed by
   an observed future result. A stronger version would use a
   forward-looking label (e.g., "declined over the next 30 days").

2. **This is a small anonymized slice (30,000 rows), not the full
   warehouse (79M+ rows).** Any pattern found here needs to be re-verified
   at full warehouse scale before being trusted as a general finding.

3. **No causal claims possible.** Even if a page is correctly flagged as
   declining, this data can't tell me whether refreshing it would actually
   cause a recovery — that would need a controlled experiment, which this
   dataset doesn't provide.

4. **Unbalanced history isn't directly visible in the starter slice**
   (this matters more at full warehouse scale, where different clients
   have different amounts of tracking history — something I'll need to
   check via `dim_clients.gsc_data_start` / `ga4_data_start` if I move to
   the warehouse release later).

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] Submitted the repo URL on the assignment card